# Commercial investigation vertical extraction

Load the **commercial_investigation** subset from intent-classified data, then extract **business verticals** (e.g. sports, health, real estate) using:
1. **IAB Content Taxonomy** keyword matching (all queries + all answers)
2. **LLM-based** structured output with intent re-classification (parallel, cached)
3. **Embedding-based** similarity to reference verticals
4. **Rule-based** vertical keywords (optional)

Uses **all user messages (all queries)** and **all assistant messages (all answers)** per conversation.

In [ ]:
# Control variables (edit and run first)
INTENT_OUTPUT_DIR = "intent_output"
USE_SAMPLE = True
SAMPLE_N = 2000
RANDOM_SEED = 42
RUN_TAXONOMY = True
RUN_LLM_VERTICAL = True
RUN_EMBEDDING_VERTICAL = True
RUN_RULE_VERTICAL = True
IAB_DATA_DIR = None   # e.g. Path("data") to use data/iab_content_taxonomy_tier1_tier2.csv
VERTICAL_LLM_CACHE_PATH = "intent_output/vertical_intent_llm.parquet"
LLM_BATCH_SIZE = 100
LLM_MAX_WORKERS = 10

## Load commercial_investigation and add text columns

Load the commercial_investigation parquet, ensure conversation is parsed, optionally sample, then add **all_queries** and **all_answers** (all user and all assistant messages per conversation).

In [2]:
from pathlib import Path
import pandas as pd

from eda_utils import ensure_conversation_parsed
from intent_analysis_utils import ensure_conversation_normalized
from intent_taxonomy import load_category
from commercial_vertical_utils import add_all_queries_answers_columns

df = load_category(INTENT_OUTPUT_DIR, "commercial_investigation", which="major")
df = ensure_conversation_parsed(df)
df = ensure_conversation_normalized(df)
if USE_SAMPLE and len(df) > SAMPLE_N:
    df = df.sample(n=min(SAMPLE_N, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
df = add_all_queries_answers_columns(df, conversation_col="conversation")
print(f"Loaded {len(df)} commercial_investigation rows. Columns: {list(df.columns)}")
df[["conversation_id", "all_queries", "all_answers"]].head(2)

Loaded 200 commercial_investigation rows. Columns: ['conversation_id', 'model', 'timestamp', 'conversation', 'turn', 'language', 'openai_moderation', 'detoxify_moderation', 'toxic', 'redacted', 'text', 'intent_major', 'intent_sub', 'all_queries', 'all_answers']


,conversation_id,all_queries,all_answers
0,dd7456f94191d7684daf0309055c277d,create a humor immaculately detailed Scott the...,"I'm sorry, I cannot generate inappropriate or ..."
1,3e7bdb47db4b98df9d4d2fae1191eed0,If a female friend does wear a dress size 8. I...,It is difficult to accurately estimate someone...


## Taxonomy-based vertical (IAB keyword match)

Match keywords from **all_queries + all_answers** to IAB Content Taxonomy (Tier 1 / Tier 2). Uses local CSV in `data/` if `IAB_DATA_DIR` is set, otherwise fetches from IAB GitHub or uses embedded fallback.

In [3]:
if RUN_TAXONOMY:
    from commercial_vertical_utils import assign_vertical_iab

    data_dir = Path(IAB_DATA_DIR) if IAB_DATA_DIR else None
    df = assign_vertical_iab(
        df,
        text_col_queries="all_queries",
        text_col_answers="all_answers",
        data_dir=data_dir,
    )
    print("Vertical (IAB) Tier 1 value counts:")
    display(df["vertical_tier1_iab"].value_counts().head(15))

Vertical (IAB) Tier 1 value counts:


vertical_tier1_iab
Family and Relationships               71
Personal Celebrations & Life Events    22
Sensitive Topics                       18
Style & Fashion                        14
Home & Garden                          14
Food & Drink                           12
Real Estate                            11
Sports                                  6
Healthy Living                          5
Books and Literature                    5
Education                               4
Automotive                              3
Fine Art                                3
Hobbies & Interests                     3
Science                                 2
Name: count, dtype: int64

## LLM-based vertical and intent re-classification

Call OpenAI with **all_queries + all_answers** per conversation; get structured JSON with `vertical_tier1`, `vertical_tier2`, and `intent_revised`. Runs in parallel with caching.

In [4]:
if RUN_LLM_VERTICAL:
    from dotenv import load_dotenv
    load_dotenv()

    from commercial_vertical_utils import label_vertical_intent_llm_parallel

    df = label_vertical_intent_llm_parallel(
        df,
        queries_col="all_queries",
        answers_col="all_answers",
        cache_path=VERTICAL_LLM_CACHE_PATH,
        use_cache=True,
        batch_size=LLM_BATCH_SIZE,
        max_workers=LLM_MAX_WORKERS,
    )
    print("Vertical (LLM) Tier 1 value counts:")
    display(df["vertical_tier1_llm"].value_counts().head(15))
    print("Intent revised vs intent_major:")
    display(pd.crosstab(df["intent_major"], df["intent_revised"], margins=True))

Vertical (LLM) Tier 1 value counts:


vertical_tier1_llm
Other           74
Technology      38
Education       30
Health          23
Finance          9
Sports           8
Shopping         6
Food & Drink     4
Travel           3
Automotive       2
Art              1
Marketing        1
Real Estate      1
Name: count, dtype: int64

Intent revised vs intent_major:


intent_revised,commercial_investigation,informational,navigational,transactional,All
intent_major,,,,,
commercial_investigation,7,191,1,1,200
All,7,191,1,1,200


## Embedding-based vertical

Embed **all_queries + all_answers** and assign the reference vertical with highest cosine similarity.

In [5]:
if RUN_EMBEDDING_VERTICAL:
    from commercial_vertical_utils import assign_vertical_embedding

    df = assign_vertical_embedding(
        df,
        text_col_queries="all_queries",
        text_col_answers="all_answers",
    )
    print("Vertical (embedding) value counts:")
    display(df["vertical_embedding"].value_counts().head(15))

/Users/Larry.Jin/miniconda3/envs/wildchat/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1623.35it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2170.57it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
-------------------

Vertical (embedding) value counts:


vertical_embedding
Health          36
Technology      27
Other           23
Education       21
Sports          18
Finance         18
Shopping        18
Food & Drink    16
Travel          10
Automotive       9
Real Estate      4
Name: count, dtype: int64

## Rule-based vertical (optional)

Assign vertical from curated keyword rules (first match wins) over **all_queries + all_answers**.

In [6]:
if RUN_RULE_VERTICAL:
    from commercial_vertical_utils import assign_vertical_rule_based

    df = assign_vertical_rule_based(
        df,
        text_col_queries="all_queries",
        text_col_answers="all_answers",
    )
    print("Vertical (rule) value counts:")
    display(df["vertical_rule"].value_counts().head(15))

Vertical (rule) value counts:


vertical_rule
Sports         81
Real Estate    52
Technology     29
Health         19
Shopping        6
Automotive      5
Other           4
Finance         2
Travel          2
Name: count, dtype: int64

## Summary: counts and agreement

Compare vertical distributions across methods and intent_revised vs intent_major.

In [7]:
summary_cols = ["vertical_tier1_iab", "vertical_tier1_llm", "vertical_embedding", "vertical_rule"]
existing = [c for c in summary_cols if c in df.columns]
if existing:
    for col in existing:
        print(f"--- {col} ---")
        display(df[col].value_counts().head(10))
if "intent_revised" in df.columns:
    print("Intent revised vs intent_major (crosstab):")
    display(pd.crosstab(df["intent_major"], df["intent_revised"], normalize="index").round(2))

--- vertical_tier1_iab ---


vertical_tier1_iab
Family and Relationships               71
Personal Celebrations & Life Events    22
Sensitive Topics                       18
Style & Fashion                        14
Home & Garden                          14
Food & Drink                           12
Real Estate                            11
Sports                                  6
Healthy Living                          5
Books and Literature                    5
Name: count, dtype: int64

--- vertical_tier1_llm ---


vertical_tier1_llm
Other           74
Technology      38
Education       30
Health          23
Finance          9
Sports           8
Shopping         6
Food & Drink     4
Travel           3
Automotive       2
Name: count, dtype: int64

--- vertical_embedding ---


vertical_embedding
Health          36
Technology      27
Other           23
Education       21
Sports          18
Finance         18
Shopping        18
Food & Drink    16
Travel          10
Automotive       9
Name: count, dtype: int64

--- vertical_rule ---


vertical_rule
Sports         81
Real Estate    52
Technology     29
Health         19
Shopping        6
Automotive      5
Other           4
Finance         2
Travel          2
Name: count, dtype: int64

Intent revised vs intent_major (crosstab):


intent_revised,commercial_investigation,informational,navigational,transactional
intent_major,,,,
commercial_investigation,0.04,0.96,0.0,0.0
